# Notebook 05 — Review Volume Trends

## 1. Setup

In [1]:
import sys
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from pathlib import Path
from IPython.display import display, HTML

PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

INTERIM_DIR  = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR  = PROJECT_ROOT / "outputs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Environment ready.")

Environment ready.


## 2. Load data

In [2]:
df     = pd.read_parquet(INTERIM_DIR / "analysis_ready.parquet")
sbux   = df[df["brand_category"] == "Starbucks"]
market = df[df["brand_category"] != "Starbucks"]

print(f"Total reviews  : {len(df):,}")
print(f"Starbucks      : {len(sbux):,}")
print(f"Market         : {len(market):,}")
print(f"Date range     : {df['date'].min().date()} -> {df['date'].max().date()}")

Total reviews  : 381,999
Starbucks      : 11,675
Market         : 370,324
Date range     : 2017-01-01 -> 2021-12-31


## 3. Annual review volume

In [3]:
sbux_yr = sbux.groupby("year")["review_id"].count().reset_index()
sbux_yr.columns = ["year", "reviews"]

print("Starbucks annual review volume:")
display(sbux_yr)

Starbucks annual review volume:


,year,reviews
0,2017,2536
1,2018,2719
2,2019,2996
3,2020,1671
4,2021,1753


In [4]:
display(sbux_yr)

# Calculate YoY % change
sbux_yr["yoy_pct"] = sbux_yr["reviews"].pct_change() * 100

fig = go.Figure()

# Bar chart (primary y-axis)
fig.add_trace(go.Bar(
    x=sbux_yr["year"].astype(str),
    y=sbux_yr["reviews"],
    text=sbux_yr["reviews"],
    textposition="outside",
    marker_color="#00704A",
    marker_line_color="#004d2e",
    marker_line_width=1,
    name="Reviews",
    yaxis="y",
))

# Line chart - YoY % change (secondary y-axis)
fig.add_trace(go.Scatter(
    x=sbux_yr["year"].astype(str),
    y=sbux_yr["yoy_pct"],
    mode="lines+markers+text",
    text=[f"{v:+.1f}%" if pd.notna(v) else "" for v in sbux_yr["yoy_pct"]],
    textposition="bottom center",
    textfont=dict(size=11, color="#e63946"),
    line=dict(color="#e63946", width=2.5),
    marker=dict(size=8, color="#e63946"),
    name="YoY %",
    yaxis="y2",
))

fig.update_layout(
    title=dict(text="Starbucks — Annual Review Volume (2017–2021)", font=dict(size=16)),
    xaxis_title="Year",
    yaxis=dict(
        title="Review Count",
        showgrid=True, gridcolor="#eeeeee",
        range=[0, 3500],
    ),
    yaxis2=dict(
        title="YoY Change %",
        overlaying="y",
        side="right",
        showgrid=False,
        zeroline=True, zerolinecolor="#cccccc", zerolinewidth=1,
        range=[-55, 25],
    ),
    plot_bgcolor="white",
    paper_bgcolor="white",
    xaxis=dict(showgrid=False),
    width=720, height=460,
    margin=dict(t=60, b=50, l=60, r=60),
    showlegend=True,
    legend=dict(x=0.01, y=0.95),
)
fig.write_html(str(FIGURES_DIR / "05_volume_annual.html"))
fig.show()

,year,reviews
0,2017,2536
1,2018,2719
2,2019,2996
3,2020,1671
4,2021,1753


## 4. Quarterly review volume

In [5]:
sbux_copy = sbux.copy()
sbux_copy["quarter_label"] = "Q" + sbux_copy["quarter"].astype(str)
vol = sbux_copy.groupby(["year", "quarter_label"])["review_id"].count().reset_index()
vol.columns = ["year", "quarter", "reviews"]

print("Starbucks quarterly volume:")
display(vol)

Starbucks quarterly volume:


,year,quarter,reviews
0,2017,Q1,650
1,2017,Q2,662
2,2017,Q3,606
3,2017,Q4,618
4,2018,Q1,648
5,2018,Q2,672
6,2018,Q3,707
7,2018,Q4,692
8,2019,Q1,681
9,2019,Q2,779


In [6]:
display(vol)

YEAR_COLORS = {
    2017: "#b0d0e8",
    2018: "#6aaed6",
    2019: "#2171b5",
    2020: "#e63946",
    2021: "#f4a261",
}

fig = go.Figure()
for yr, grp in vol.groupby("year"):
    fig.add_trace(go.Scatter(
        x=grp["quarter"],
        y=grp["reviews"],
        mode="lines+markers",
        name=str(int(yr)),
        line=dict(color=YEAR_COLORS[int(yr)], width=2.5),
        marker=dict(size=8),
    ))

fig.update_layout(
    title=dict(text="Starbucks — Quarterly Review Volume by Year", font=dict(size=16)),
    xaxis_title="Quarter",
    yaxis_title="Review Count",
    plot_bgcolor="white",
    paper_bgcolor="white",
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=True, gridcolor="#eeeeee"),
    legend_title="Year",
    width=820, height=480,
    margin=dict(t=60, b=50, l=60, r=40)
)
fig.write_html(str(FIGURES_DIR / "05_volume_by_year.html"))
fig.show()

,year,quarter,reviews
0,2017,Q1,650
1,2017,Q2,662
2,2017,Q3,606
3,2017,Q4,618
4,2018,Q1,648
5,2018,Q2,672
6,2018,Q3,707
7,2018,Q4,692
8,2019,Q1,681
9,2019,Q2,779


## 5. Year-over-year comparison (Starbucks vs. market)

In [7]:
sbux_yr_ct  = sbux.groupby("year")["review_id"].count()
mkt_yr_ct   = market.groupby("year")["review_id"].count()
sbux_yoy    = sbux_yr_ct.pct_change() * 100
mkt_yoy     = mkt_yr_ct.pct_change() * 100

yoy = pd.DataFrame({
    "Starbucks Reviews": sbux_yr_ct,
    "Starbucks YoY %":  sbux_yoy.round(1),
    "Market Reviews":   mkt_yr_ct,
    "Market YoY %":     mkt_yoy.round(1),
})
display(yoy)

,Starbucks Reviews,Starbucks YoY %,Market Reviews,Market YoY %
year,,,,
2017,2536,NaN,81423,NaN
2018,2719,7.2,89393,9.8
2019,2996,10.2,87706,-1.9
2020,1671,-44.2,51847,-40.9
2021,1753,4.9,59955,15.6


## 6. Geographic distribution

In [8]:
state_vol = sbux.groupby("state")["review_id"].count().reset_index()
state_vol.columns = ["state", "reviews"]
state_vol = state_vol.sort_values("reviews", ascending=False)

print("Review count by state:")
display(state_vol)

Review count by state:


,state,reviews
3,FL,2061
11,PA,2042
6,IN,1333
10,NV,1155
12,TN,1009
8,MO,914
0,AZ,906
7,LA,753
1,CA,483
9,NJ,447


In [9]:
display(state_vol)

# State centroid coordinates for text labels
STATE_COORDS = {
    "PA": (40.9, -77.8), "FL": (28.6, -82.4), "LA": (31.0, -92.0),
    "TN": (35.9, -86.4), "IN": (39.8, -86.3), "MO": (38.4, -92.5),
    "NJ": (40.1, -74.7), "AZ": (34.2, -111.7), "NV": (39.4, -117.0),
    "ID": (44.4, -114.6), "CA": (36.8, -119.4), "DE": (39.0, -75.5),
    "IL": (40.0, -89.2),
}

fig = px.choropleth(
    state_vol,
    locations="state",
    locationmode="USA-states",
    color="reviews",
    scope="usa",
    color_continuous_scale="YlOrRd",
    title="Starbucks — Review Volume by State (2017–2021)",
    labels={"reviews": "Reviews"}
)

# Add state abbreviation labels
fig.add_scattergeo(
    locations=state_vol["state"],
    locationmode="USA-states",
    text=state_vol["state"],
    mode="text",
    textfont=dict(size=10, color="black", family="Arial Black"),
    showlegend=False,
    lat=[STATE_COORDS.get(s, (0,0))[0] for s in state_vol["state"]],
    lon=[STATE_COORDS.get(s, (0,0))[1] for s in state_vol["state"]],
)

fig.update_layout(
    width=900, height=520,
    paper_bgcolor="white",
    margin=dict(t=60, b=20, l=20, r=20),
    coloraxis_colorbar=dict(title="Reviews")
)
fig.write_html(str(FIGURES_DIR / "05_volume_by_state.html"))
fig.show()

,state,reviews
3,FL,2061
11,PA,2042
6,IN,1333
10,NV,1155
12,TN,1009
8,MO,914
0,AZ,906
7,LA,753
1,CA,483
9,NJ,447


In [10]:
# State-level average star rating for Starbucks
state_stars = sbux.groupby("state")["review_stars"].mean().reset_index()
state_stars.columns = ["state", "avg_stars"]
state_stars["avg_stars"] = state_stars["avg_stars"].round(2)

display(state_stars.sort_values("avg_stars", ascending=False))

fig_s = px.choropleth(
    state_stars,
    locations="state",
    locationmode="USA-states",
    color="avg_stars",
    scope="usa",
    color_continuous_scale="RdYlGn",
    range_color=[2.0, 4.0],
    title="Starbucks — Avg Star Rating by State (2017–2021)",
    labels={"avg_stars": "Avg ★"}
)

# Add state abbreviation labels
fig_s.add_scattergeo(
    locations=state_stars["state"],
    locationmode="USA-states",
    text=state_stars["state"],
    mode="text",
    textfont=dict(size=10, color="black", family="Arial Black"),
    showlegend=False,
    lat=[STATE_COORDS.get(s, (0,0))[0] for s in state_stars["state"]],
    lon=[STATE_COORDS.get(s, (0,0))[1] for s in state_stars["state"]],
)

fig_s.update_layout(
    width=900, height=520,
    paper_bgcolor="white",
    margin=dict(t=60, b=20, l=20, r=20),
    coloraxis_colorbar=dict(title="Avg ★"),
)
fig_s.write_html(str(FIGURES_DIR / "05_rating_by_state.html"))
fig_s.show()

,state,avg_stars
6,IN,3.28
4,ID,3.15
3,FL,3.06
0,AZ,3.04
5,IL,2.92
8,MO,2.86
9,NJ,2.86
10,NV,2.80
1,CA,2.79
11,PA,2.79


## Key Findings

- Review volume grew from 2,536 (2017) to 2,996 (2019), with YoY gains of +7.2% and +10.2%. Peak quarter was Q2 2019 at 779 reviews.
- COVID-19 caused a -44.2% drop in 2020 (1,671 reviews). Q2 2020 hit a floor of 222 reviews.
- 2021 recovered only to 1,753 reviews, 58.5% of the 2019 peak. YoY rebound was +4.9%.
- Top states by volume: FL (2,061), PA (2,042), IN (1,333), NV (1,155), TN (1,009) across 508 cities in 13 states.
- Top states by rating: IN (3.28), ID (3.15), FL (3.06), AZ (3.04). High volume doesn't guarantee high satisfaction.